In [1]:
%pip install -q kagglehub soundfile scipy



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: pip3.12 install --upgrade pip
Note: you may need to restart the kernel to use updated packages.


In [2]:
import shutil
from pathlib import Path
import kagglehub
# ===== User-configurable paths/toggles =====
PROJECT_NAME = "mert-music-retrieval"
LOCAL_PROJECT_ROOT = Path("/Users/samuelturner/Documents/mert-music-retrieval")
COLAB_DRIVE_ROOT = Path("/content/drive/MyDrive")
SAVE_TO_DRIVE_WHEN_COLAB = True   # Turn on/off Drive saving when running in Colab
OVERWRITE_EXISTING = False         # If True, replace files already in target
# ===== Detect environment =====
try:
    import google.colab  # type: ignore
    IN_COLAB = True
except Exception:
    IN_COLAB = False
# ===== Resolve target path =====
if IN_COLAB and SAVE_TO_DRIVE_WHEN_COLAB:
    from google.colab import drive  # type: ignore
    drive.mount('/content/drive', force_remount=False)
    PROJECT_ROOT = COLAB_DRIVE_ROOT / PROJECT_NAME
else:
    PROJECT_ROOT = LOCAL_PROJECT_ROOT
TARGET_ROOT = PROJECT_ROOT / 'data' / 'raw' / 'gtzan'
TARGET_ROOT.mkdir(parents=True, exist_ok=True)
print('IN_COLAB =', IN_COLAB)
print('PROJECT_ROOT =', PROJECT_ROOT)
print('TARGET_ROOT =', TARGET_ROOT)
# ===== Download GTZAN from Kaggle =====
dataset_path = Path(kagglehub.dataset_download('andradaolteanu/gtzan-dataset-music-genre-classification'))
print('Downloaded to cache:', dataset_path)
source_genres = dataset_path / 'Data' / 'genres_original'
if not source_genres.exists():
    raise FileNotFoundError(f'Expected folder not found: {source_genres}')
# ===== Copy audio into project data root =====
copied = 0
skipped = 0
for genre_dir in sorted(source_genres.iterdir()):
    if not genre_dir.is_dir():
        continue
    out_dir = TARGET_ROOT / genre_dir.name
    out_dir.mkdir(parents=True, exist_ok=True)
    for wav in sorted(genre_dir.glob('*.wav')):
        dest = out_dir / wav.name
        if dest.exists() and not OVERWRITE_EXISTING:
            skipped += 1
            continue
        shutil.copy2(wav, dest)
        copied += 1
total_wavs = len(list(TARGET_ROOT.glob('*/*.wav')))
print('Copied:', copied)
print('Skipped existing:', skipped)
print('Total wav files now in target:', total_wavs)


IN_COLAB = False
PROJECT_ROOT = /Users/samuelturner/Documents/mert-music-retrieval
TARGET_ROOT = /Users/samuelturner/Documents/mert-music-retrieval/data/raw/gtzan


/Library/Frameworks/Python.framework/Versions/3.12/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Downloaded to cache: /Users/samuelturner/.cache/kagglehub/datasets/andradaolteanu/gtzan-dataset-music-genre-classification/versions/1
Copied: 0
Skipped existing: 1000
Total wav files now in target: 1000


In [3]:
# Decode preprocess audit (validate files before 5s clipping)
import runpy
from pathlib import Path

if 'PROJECT_ROOT' not in globals():
    PROJECT_NAME = globals().get('PROJECT_NAME', 'mert-music-retrieval')
    LOCAL_PROJECT_ROOT = Path(globals().get('LOCAL_PROJECT_ROOT', '/Users/samuelturner/Documents/mert-music-retrieval'))
    COLAB_DRIVE_ROOT = Path(globals().get('COLAB_DRIVE_ROOT', '/content/drive/MyDrive'))
    SAVE_TO_DRIVE_WHEN_COLAB = globals().get('SAVE_TO_DRIVE_WHEN_COLAB', True)

    try:
        import google.colab  # type: ignore
        IN_COLAB = True
    except Exception:
        IN_COLAB = False

    if IN_COLAB and SAVE_TO_DRIVE_WHEN_COLAB:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive', force_remount=False)
        PROJECT_ROOT = COLAB_DRIVE_ROOT / PROJECT_NAME
    else:
        PROJECT_ROOT = LOCAL_PROJECT_ROOT

import os

# Pass resolved root into scripts
os.environ['MMR_PROJECT_ROOT'] = str(PROJECT_ROOT)

decode_script = PROJECT_ROOT / 'scripts' / 'decode_preprocess_gtzan.py'
if not decode_script.exists():
    raise FileNotFoundError(f'Decode script not found at: {decode_script}')

runpy.run_path(str(decode_script), run_name='__main__')

decode_report = PROJECT_ROOT / 'artifacts' / 'decode_preprocess_report.md'
print(decode_report.read_text())


Wrote: /Users/samuelturner/Documents/mert-music-retrieval/artifacts/decode_preprocess_report.md
# Decode Preprocess Report

- Input tracks: 1000
- Output audit: `/Users/samuelturner/Documents/mert-music-retrieval/artifacts/decode_preprocess_audit.csv`

## Decode Status Counts
- decode_error: 1
- ok: 999

## Split Decode Coverage
- train: 799/800 ok
- val: 100/100 ok
- test: 100/100 ok



In [4]:
# Build 5-second clip dataset from track-level splits (same Drive/local logic)
import runpy
from pathlib import Path

# Reuse PROJECT_ROOT from previous cell if available; otherwise resolve with same logic
if 'PROJECT_ROOT' not in globals():
    PROJECT_NAME = globals().get('PROJECT_NAME', 'mert-music-retrieval')
    LOCAL_PROJECT_ROOT = Path(globals().get('LOCAL_PROJECT_ROOT', '/Users/samuelturner/Documents/mert-music-retrieval'))
    COLAB_DRIVE_ROOT = Path(globals().get('COLAB_DRIVE_ROOT', '/content/drive/MyDrive'))
    SAVE_TO_DRIVE_WHEN_COLAB = globals().get('SAVE_TO_DRIVE_WHEN_COLAB', True)

    try:
        import google.colab  # type: ignore
        IN_COLAB = True
    except Exception:
        IN_COLAB = False

    if IN_COLAB and SAVE_TO_DRIVE_WHEN_COLAB:
        from google.colab import drive  # type: ignore
        drive.mount('/content/drive', force_remount=False)
        PROJECT_ROOT = COLAB_DRIVE_ROOT / PROJECT_NAME
    else:
        PROJECT_ROOT = LOCAL_PROJECT_ROOT

print('PROJECT_ROOT =', PROJECT_ROOT)
import os

# Pass resolved root into scripts
os.environ['MMR_PROJECT_ROOT'] = str(PROJECT_ROOT)

script_path = PROJECT_ROOT / 'scripts' / 'preprocess_gtzan_5s.py'
if not script_path.exists():
    raise FileNotFoundError(f'Preprocess script not found at: {script_path}')

# Self-healing import in case notebook state gets weird
if 'runpy' not in globals():
    import runpy

runpy.run_path(str(script_path), run_name='__main__')

summary_path = PROJECT_ROOT / 'artifacts' / 'preprocessing_5s_summary.md'
print(summary_path.read_text())


PROJECT_ROOT = /Users/samuelturner/Documents/mert-music-retrieval
Wrote: /Users/samuelturner/Documents/mert-music-retrieval/artifacts/preprocessing_5s_summary.md
# 5s Preprocessing Summary

- Target sample rate: 24000
- Clip seconds: 5.0
- Output audio root: `/Users/samuelturner/Documents/mert-music-retrieval/data/interim/gtzan_5s`
- Output manifest root: `/Users/samuelturner/Documents/mert-music-retrieval/artifacts/splits_5s`

## train
- Input tracks: 800
- Output clips: 5584
- Failed tracks: 1
- Tracks resampled to 24000: 799
- Tracks amplitude-normalized: 799
- Tracks mono-folded from multi-channel: 0
- Original sample rates seen: {22050: 799}

## val
- Input tracks: 100
- Output clips: 700
- Failed tracks: 0
- Tracks resampled to 24000: 100
- Tracks amplitude-normalized: 100
- Tracks mono-folded from multi-channel: 0
- Original sample rates seen: {22050: 100}

## test
- Input tracks: 100
- Output clips: 699
- Failed tracks: 0
- Tracks resampled to 24000: 100
- Tracks amplitude-norm